In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import os
from google.colab import drive #type:ignore
import torch

drive.mount(r'/content/drive/')
df = pd.read_csv(r"/content/drive/MyDrive/DL_CSV/fmnist_small.csv")
x = df.iloc[:,1:].values
y = df.iloc[:,0].values

# Checking for the availability of GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

x = x / 255.0

print(x)
print(y)

# Creating custom class dataset
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32) 
        self.labels = torch.tensor(labels, dtype=torch.long) 

    def __len__(self):
        return len(self.features)     

    def __getitem__(self, index):
        return self.features[index], self.labels[index]
    
# Performing DataSet Split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, shuffle=True)

# Creating CustomDataset Object for train and test
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

# Creating DataLoader object of the class
# Note: num_workers=2 works fine, but if Colab ever throws a broken pipe error, change it to 0.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,num_workers=0)

# Creating our Artificial Neural Network
class ArtificialNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, features):
        return self.network(features)
        
# Defining Epochs and Learning Rate
epochs = 100
learning_rate = 0.1

model = ArtificialNN(X_train.shape[1])
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# Performing Training
for i in range(epochs):
    total_loss = 0

    for features, label in train_loader:
        features, label = features.to(device), label.to(device)
        y_pred = model(features)

        # Making gradient zero
        optimizer.zero_grad()

        # Calculating Loss
        loss = loss_function(y_pred, label)

        # Backward
        loss.backward()

        # Updating Parameters
        optimizer.step()

        total_loss = total_loss + loss.item()
    
    print(f"For epoch :{i+1}, Loss={total_loss/len(train_loader)}")

# Evaluating the model
model.eval() 
total = 0
correct = 0

with torch.no_grad():
    for features, labels in test_loader:
        # FIX: Changed labels(device) to labels.to(device)
        features, labels = features.to(device), labels.to(device)
        y_pred = model(features)

        _, predicted = torch.max(y_pred, 1)
        total = total + labels.shape[0]

        correct = correct + (predicted == labels).sum().item()

print(f"Accuracy of Model = {(correct/total)*100:.2f}%")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
Using device: cuda
[[0.         0.         0.         ... 0.64705882 0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
[9 7 0 ... 8 4 8]
For epoch :1, Loss=1.3438077437877656
For epoch :2, Loss=0.7954437057177226
For epoch :3, Loss=0.678323648571968
For epoch :4, Loss=0.5924979817867279
For epoch :5, Loss=0.5556119934717814
For epoch :6, Loss=0.5097244171301524
For epoch :7, Loss=0.4730639957388242
For epoch :8, Loss=0.46080111905932425
For epoch :9, Loss=0.42117338716983793
For epoch :10, Loss=0.4162323490778605
For ep